# 将 Azure AI Search 用作向量数据库，并结合 Azure Functions 在 ChatGPT 中集成 GPT

本 notebook 分步介绍如何将 Azure AI Search（前身为 Azure Cognitive Search）用作存储 OpenAI 嵌入的向量数据库，然后在其之上创建一个 Azure Function，以接入 ChatGPT 中的自定义 GPT（Custom GPT）。

对于希望在 Azure 内部搭建 RAG 基础设施、并将其作为端点对外暴露以便与 ChatGPT 等其他平台集成的客户来说，这可以作为一种解决方案。

Azure AI Search 是一项云搜索服务，为开发者提供基础设施、API 和工具，帮助他们在 Web、移动和企业应用中，针对私有的异构内容构建丰富的搜索体验。

Azure Functions 是一种无服务器计算服务，用于运行事件驱动的代码，能够自动管理基础设施和扩缩容，并与其他 Azure 服务集成。

## 前提条件：
完成本练习需要具备以下条件：
- 具有创建 [Azure AI Search 服务](https://learn.microsoft.com/azure/search/)和 Azure Function App 权限的 Azure 用户
- Azure 订阅 ID 和一个资源组。
- [OpenAI 密钥](https://platform.openai.com/account/api-keys)

# 架构
下图展示了该解决方案的架构，我们将逐步进行讲解。

![azure-rag-architecture.png](../../../../images/azure-rag-architecture.png)


> 注意：这种“向量数据存储 + 无服务器函数”的架构模式也可以推广到其他向量数据存储。例如，如果你想在 Azure 中使用 Postgres 之类的数据库，可以将[配置 Azure AI Search 设置](#配置-azure-ai-search-设置)这一步改为设置 Postgres 所需的条件，将[创建 Azure AI 向量搜索](#创建-azure-ai-向量搜索)改为在 Postgres 中创建数据库和表，并更新本仓库中的 `function_app.py` 代码，改为查询 Postgres 而不是 Azure AI Search。数据准备和 Azure Function 的创建则保持不变。


# 目录：

1. **[设置环境](#设置环境)**
    通过安装和导入所需的库并配置 Azure 设置来准备环境。包括：
     - [安装并导入所需的库](#安装并导入所需的库)
     - [配置 OpenAI 设置](#配置-openai-设置)
     - [配置 Azure AI Search 设置](#配置-azure-ai-search-设置)
 

2. **[准备数据](#准备数据)** 通过对文档进行嵌入并收集额外的元数据，为上传准备数据。我们将使用 OpenAI 文档的一个子集作为示例数据。
 
3. **[创建 Azure AI 向量搜索](#创建-azure-ai-向量搜索)** 创建 Azure AI 向量搜索并上传我们准备好的数据。包括：
     - [创建索引](#创建索引)：在 Azure AI Search 中创建索引的步骤。
     - [上传数据](#上传数据)：将数据上传到 Azure AI Search 的说明。
     - [测试搜索](#测试搜索)：测试搜索功能的步骤。
 
4. **[创建 Azure 函数](#创建-azure-函数)** 创建一个与 Azure AI 向量搜索交互的 Azure Function。包括：
     - [创建存储账户](#创建存储账户)：为 Azure Function 创建存储账户的步骤。
     - [创建 Function App](#创建-function-app)：在 Azure 中创建 Function App 的说明。
 
5. **[在 ChatGPT 的自定义 GPT 中接入](#在-chatgpt-的自定义-gpt-中接入)** 将 Azure Function 与 ChatGPT 中的自定义 GPT 集成。包括：
     - [创建 OpenAPI 规范](#创建-openapi-规范)：为 Azure Function 创建 OpenAPI 规范的步骤。
     - [创建 GPT 指令](#创建-gpt-指令)：为此集成编写 GPT 专用指令的说明。




# 设置环境
我们将通过导入所需的库并配置 Azure 设置来准备环境。

## 安装并导入所需的库
为了便于阅读，我们将这些库分为 Python 标准库、第三方库以及 Azure 相关库。

In [ ]:
! pip install -q wget
! pip install -q azure-search-documents 
! pip install -q azure-identity
! pip install -q openai
! pip install -q azure-mgmt-search
! pip install -q pandas
! pip install -q azure-mgmt-resource 
! pip install -q azure-mgmt-storage
! pip install -q pyperclip
! pip install -q PyPDF2
! pip install -q tiktoken

In [ ]:
# Standard Libraries
import json  
import os
import platform
import subprocess
import csv
from itertools import islice
import uuid
import shutil
import concurrent.futures

# Third-Party Libraries
import pandas as pd
from PyPDF2 import PdfReader
import tiktoken
from dotenv import load_dotenv
import pyperclip

# OpenAI Libraries (note we use OpenAI directly here, but you can replace with Azure OpenAI as needed)
from openai import OpenAI

# Azure Identity and Credentials
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.core.credentials import AzureKeyCredential  
from azure.core.exceptions import HttpResponseError

# Azure Search Documents
from azure.search.documents import SearchClient, SearchIndexingBufferedSender  
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.models import (
    VectorizedQuery
)
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    HnswParameters,
    SearchField,
    SearchableField,
    SearchFieldDataType,
    SearchIndex,
    SimpleField,
    VectorSearch,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

# Azure Management Clients
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.resource import ResourceManagementClient, SubscriptionClient
from azure.mgmt.storage import StorageManagementClient

## 配置 OpenAI 设置

开始本节之前，请确保你已准备好 OpenAI API 密钥。


In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as an env var>") # Saving this as a variable to reference in function app in later step
openai_client = OpenAI(api_key=openai_api_key)
embeddings_model = "text-embedding-3-small" # We'll use this by default, but you can change to your text-embedding-3-large if desired

## 配置 Azure AI Search 设置
你可以在 Azure 门户中找到 Azure AI Search 服务的详细信息，也可以通过 [Search Management SDK](https://learn.microsoft.com/rest/api/searchmanagement/) 以编程方式获取。


#### 前提条件：
- Azure 订阅 ID
- Azure 资源组名称
- Azure 区域

In [ ]:
# Update the below with your values
subscription_id="<enter_your_subscription_id>"
resource_group="<enter_your_resource_group>"

## Make sure to choose a region that supports the proper products. We've defaulted to "eastus" below. https://azure.microsoft.com/en-us/explore/global-infrastructure/products-by-region/#products-by-region_tab5
region = "eastus"
credential = InteractiveBrowserCredential()
subscription_client = SubscriptionClient(credential)
subscription = next(subscription_client.subscriptions.list())

#### 创建并配置 Azure AI Search 服务
下面我们将为搜索服务生成一个唯一名称，设置服务属性，并创建搜索服务。

In [ ]:
# Initialize the SearchManagementClient with the provided credentials and subscription ID
search_management_client = SearchManagementClient(
    credential=credential,
    subscription_id=subscription_id,
)

# Generate a unique name for the search service using UUID, but you can change this if you'd like.
generated_uuid = str(uuid.uuid4())
search_service_name = "search-service-gpt-demo" + generated_uuid
## The below is the default endpoint structure that is created when you create a search service. This may differ based on your Azure settings.
search_service_endpoint = 'https://'+search_service_name+'.search.windows.net'

# Create or update the search service with the specified parameters
response = search_management_client.services.begin_create_or_update(
    resource_group_name=resource_group,
    search_service_name=search_service_name,
    service={
        "location": region,
        "properties": {"hostingMode": "default", "partitionCount": 1, "replicaCount": 1},
        # We are using the free pricing tier for this demo. You are only allowed one free search service per subscription.
        "sku": {"name": "free"},
        "tags": {"app-name": "Search service demo"},
    },
).result()

# Convert the response to a dictionary and then to a pretty-printed JSON string
response_dict = response.as_dict()
response_json = json.dumps(response_dict, indent=4)

print(response_json)
print("Search Service Name:" + search_service_name)
print("Search Service Endpoint:" + search_service_endpoint)


#### 获取搜索服务 API 密钥
搜索服务启动并运行后，我们需要获取[搜索服务 API 密钥](https://learn.microsoft.com/en-us/azure/search/search-security-api-keys?tabs=rest-use,portal-find,portal-query)，用于发起索引创建，以及之后执行搜索。

In [ ]:
# Retrieve the admin keys for the search service
try:
    response = search_management_client.admin_keys.get(
        resource_group_name=resource_group,
        search_service_name=search_service_name,
    )
    # Extract the primary API key from the response and save as a variable to be used later
    search_service_api_key = response.primary_key
    print("Successfully retrieved the API key.")
except Exception as e:
    print(f"Failed to retrieve the API key: {e}")

# 准备数据
我们将对 oai_docs 文件夹中的若干 OpenAI 文档页面进行嵌入和存储。我们会先对每个页面进行嵌入并添加到 CSV 中，然后使用该 CSV 上传到索引。

为了处理超出 8191 个 token 上下文长度的较长文本文件，我们可以分别使用各个分块的嵌入，或者以某种方式将它们组合起来，例如求平均值（按每个分块的大小加权）。

我们将借用 Python 官方 cookbook 中的一个函数，用于将序列切分为多个分块。

In [ ]:
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(islice(it, n))):
        yield batch


现在我们定义一个函数，将字符串编码为 token，再将其切分为分块。我们将使用 tiktoken，这是 OpenAI 推出的一款快速开源分词器。

要进一步了解如何使用 Tiktoken 计算 token，请参阅[这篇 cookbook](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken)。



In [ ]:
def chunked_tokens(text, chunk_length, encoding_name='cl100k_base'):
    # Get the encoding object for the specified encoding name. OpenAI's tiktoken library, which is used in this notebook, currently supports two encodings: 'bpe' and 'cl100k_base'. The 'bpe' encoding is used for GPT-3 and earlier models, while 'cl100k_base' is used for newer models like GPT-4.
    encoding = tiktoken.get_encoding(encoding_name)
    # Encode the input text into tokens
    tokens = encoding.encode(text)
    # Create an iterator that yields chunks of tokens of the specified length
    chunks_iterator = batched(tokens, chunk_length)
    # Yield each chunk from the iterator
    yield from chunks_iterator

最后，我们可以编写一个函数，通过将输入 token 分块并分别对每个分块进行嵌入，即使输入文本超过最大上下文长度，也能安全地处理嵌入请求。将 average 标志设为 True 会返回各分块嵌入的加权平均值，设为 False 则直接返回未经修改的分块嵌入列表。

> 注意：这里还可以采用其他更复杂的技巧，包括：
> - 使用 GPT-4o 获取图像/图表的描述用于嵌入。
> - 在分块之间保留文本重叠，尽量避免截断重要的上下文。
> - 基于段落或章节进行分块。
> - 为每篇文章添加更具描述性的元数据。

In [ ]:
## Change the below based on model. The below is for the latest embeddings models from OpenAI, so you can leave as is unless you are using a different embedding model..
EMBEDDING_CTX_LENGTH = 8191
EMBEDDING_ENCODING='cl100k_base'

In [ ]:
def generate_embeddings(text, model):
    # Generate embeddings for the provided text using the specified model
    embeddings_response = openai_client.embeddings.create(model=model, input=text)
    # Extract the embedding data from the response
    embedding = embeddings_response.data[0].embedding
    return embedding

def len_safe_get_embedding(text, model=embeddings_model, max_tokens=EMBEDDING_CTX_LENGTH, encoding_name=EMBEDDING_ENCODING):
    # Initialize lists to store embeddings and corresponding text chunks
    chunk_embeddings = []
    chunk_texts = []
    # Iterate over chunks of tokens from the input text
    for chunk in chunked_tokens(text, chunk_length=max_tokens, encoding_name=encoding_name):
        # Generate embeddings for each chunk and append to the list
        chunk_embeddings.append(generate_embeddings(chunk, model=model))
        # Decode the chunk back to text and append to the list
        chunk_texts.append(tiktoken.get_encoding(encoding_name).decode(chunk))
    # Return the list of chunk embeddings and the corresponding text chunks
    return chunk_embeddings, chunk_texts

接下来，我们可以定义一个辅助函数，用于收集有关文档的额外元数据。这些元数据可以用作搜索查询的元数据过滤条件，也能为搜索提供更丰富的数据。

在本示例中，我将从一个类别列表中进行选择，供后续在元数据过滤中使用。

In [ ]:
## These are the categories I will be using for the categorization task. You can change these as needed based on your use case.
categories = ['authentication','models','techniques','tools','setup','billing_limits','other']

def categorize_text(text, categories):
    # Create a prompt for categorization
    messages = [
        {"role": "system", "content": f"""You are an expert in LLMs, and you will be given text that corresponds to an article in OpenAI's documentation.
         Categorize the document into one of these categories: {', '.join(categories)}. Only respond with the category name and nothing else."""},
        {"role": "user", "content": text}
    ]
    try:
        # Call the OpenAI API to categorize the text
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        # Extract the category from the response
        category = response.choices[0].message.content
        return category
    except Exception as e:
        print(f"Error categorizing text: {str(e)}")
        return None

现在，我们可以定义一些辅助函数来处理 data 文件夹下 oai_docs 文件夹中的 .txt 文件。你也可以将其用于自己的数据，它同时支持 .txt 和 .pdf 文件。

In [ ]:
def extract_text_from_pdf(pdf_path):
    # Initialize the PDF reader
    reader = PdfReader(pdf_path)
    text = ""
    # Iterate through each page in the PDF and extract text
    for page in reader.pages:
        text += page.extract_text()
    return text

def process_file(file_path, idx, categories, embeddings_model):
    file_name = os.path.basename(file_path)
    print(f"Processing file {idx + 1}: {file_name}")
    
    # Read text content from .txt files
    if file_name.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
    # Extract text content from .pdf files
    elif file_name.endswith('.pdf'):
        text = extract_text_from_pdf(file_path)
    
    title = file_name
    # Generate embeddings for the title
    title_vectors, title_text = len_safe_get_embedding(title, embeddings_model)
    print(f"Generated title embeddings for {file_name}")
    
    # Generate embeddings for the content
    content_vectors, content_text = len_safe_get_embedding(text, embeddings_model)
    print(f"Generated content embeddings for {file_name}")
    
    category = categorize_text(' '.join(content_text), categories)
    print(f"Categorized {file_name} as {category}")
    
    # Prepare the data to be appended
    data = []
    for i, content_vector in enumerate(content_vectors):
        data.append({
            "id": f"{idx}_{i}",
            "vector_id": f"{idx}_{i}",
            "title": title_text[0],
            "text": content_text[i],
            "title_vector": json.dumps(title_vectors[0]),  # Assuming title is short and has only one chunk
            "content_vector": json.dumps(content_vector),
            "category": category
        })
        print(f"Appended data for chunk {i + 1}/{len(content_vectors)} of {file_name}")
    
    return data



现在我们将使用这个辅助函数来处理 OpenAI 文档。你可以修改下面 `process_files` 中的文件夹，改用你自己的数据。

请注意，这会并发处理所选文件夹中的文档，因此如果使用 txt 文件，耗时应少于 30 秒；如果使用 PDF，则会稍长一些。

In [ ]:
## Customize the location below if you are using different data besides the OpenAI documentation. Note that if you are using a different dataset, you will need to update the categories list as well.
folder_name = "../../../data/oai_docs"

files = [os.path.join(folder_name, f) for f in os.listdir(folder_name) if f.endswith('.txt') or f.endswith('.pdf')]
data = []

# Process each file concurrently
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(process_file, file_path, idx, categories, embeddings_model): idx for idx, file_path in enumerate(files)}
    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()
            data.extend(result)
        except Exception as e:
            print(f"Error processing file: {str(e)}")

# Write the data to a CSV file
csv_file = os.path.join("..", "embedded_data.csv")
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ["id", "vector_id", "title", "text", "title_vector", "content_vector","category"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in data:
        writer.writerow(row)
        print(f"Wrote row with id {row['id']} to CSV")

# Convert the CSV file to a Dataframe
article_df = pd.read_csv("../embedded_data.csv")
# Read vectors from strings back into a list using json.loads
article_df["title_vector"] = article_df.title_vector.apply(json.loads)
article_df["content_vector"] = article_df.content_vector.apply(json.loads)
article_df["vector_id"] = article_df["vector_id"].apply(str)
article_df["category"] = article_df["category"].apply(str)
article_df.head()


现在我们得到了一个包含六列的 `embedded_data.csv` 文件，可以将其上传到向量数据库了！

# 创建 Azure AI 向量搜索

##  创建索引
我们将使用 Azure AI Search Python SDK 中的 `SearchIndexClient` 定义并创建搜索索引。该索引同时具备向量搜索和混合搜索功能。更多详情，请参阅 Microsoft 关于如何[创建向量索引](https://learn.microsoft.com/azure/search/vector-search-how-to-create-index?.tabs=config-2023-11-01%2Crest-2023-11-01%2Cpush%2Cportal-check-index)的文档

In [ ]:
index_name = "azure-ai-search-openai-cookbook-demo"
# index_name = "<insert_name_for_index>"

index_client = SearchIndexClient(
    endpoint=search_service_endpoint, credential=AzureKeyCredential(search_service_api_key)
)
# Define the fields for the index. Update these based on your data.
# Each field represents a column in the search index
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String),  # Simple string field for document ID
    SimpleField(name="vector_id", type=SearchFieldDataType.String, key=True),  # Key field for the index
    # SimpleField(name="url", type=SearchFieldDataType.String),  # URL field (commented out)
    SearchableField(name="title", type=SearchFieldDataType.String),  # Searchable field for document title
    SearchableField(name="text", type=SearchFieldDataType.String),  # Searchable field for document text
    SearchField(
        name="title_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),  # Collection of single values for title vector
        vector_search_dimensions=1536,  # Number of dimensions in the vector
        vector_search_profile_name="my-vector-config",  # Profile name for vector search configuration
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),  # Collection of single values for content vector
        vector_search_dimensions=1536,  # Number of dimensions in the vector
        vector_search_profile_name="my-vector-config",  # Profile name for vector search configuration
    ),
    SearchableField(name="category", type=SearchFieldDataType.String, filterable=True),  # Searchable field for document category
]

# This configuration defines the algorithm and parameters for vector search
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="my-hnsw",  # Name of the HNSW algorithm configuration
            kind=VectorSearchAlgorithmKind.HNSW,  # Type of algorithm
            parameters=HnswParameters(
                m=4,  # Number of bi-directional links created for every new element
                ef_construction=400,  # Size of the dynamic list for the nearest neighbors during construction
                ef_search=500,  # Size of the dynamic list for the nearest neighbors during search
                metric=VectorSearchAlgorithmMetric.COSINE,  # Distance metric used for the search
            ),
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="my-vector-config",  # Name of the vector search profile
            algorithm_configuration_name="my-hnsw",  # Reference to the algorithm configuration
        )
    ],
)

# Create the search index with the vector search configuration
# This combines all the configurations into a single search index
index = SearchIndex(
    name=index_name,  # Name of the index
    fields=fields,  # Fields defined for the index
    vector_search=vector_search  # Vector search configuration

)

# Create or update the index
# This sends the index definition to the Azure Search service
result = index_client.create_index(index)
print(f"{result.name} created")  # Output the name of the created index

## 上传数据

现在，我们将上面存储在 `embedded_data.csv` 中的文章从 pandas DataFrame 上传到 Azure AI Search 索引。有关数据导入策略和最佳实践的详细指南，请参阅 [Azure AI Search 中的数据导入](https://learn.microsoft.com/azure/search/search-what-is-data-import)。


In [ ]:
# Convert the 'id' and 'vector_id' columns to string so one of them can serve as our key field
article_df["id"] = article_df["id"].astype(str)
article_df["vector_id"] = article_df["vector_id"].astype(str)

# Convert the DataFrame to a list of dictionaries
documents = article_df.to_dict(orient="records")

# Log the number of documents to be uploaded
print(f"Number of documents to upload: {len(documents)}")

# Create a SearchIndexingBufferedSender
batch_client = SearchIndexingBufferedSender(
    search_service_endpoint, index_name, AzureKeyCredential(search_service_api_key)
)
# Get the first document to check its schema
first_document = documents[0]

# Get the index schema
index_schema = index_client.get_index(index_name)

# Get the field names from the index schema
index_fields = {field.name: field.type for field in index_schema.fields}

# Check each field in the first document
for field, value in first_document.items():
    if field not in index_fields:
        print(f"Field '{field}' is not in the index schema.")

# Check for any fields in the index schema that are not in the documents
for field in index_fields:
    if field not in first_document:
        print(f"Field '{field}' is in the index schema but not in the documents.")

try:
    if documents:
        # Add upload actions for all documents in a single call
        upload_result = batch_client.upload_documents(documents=documents)

        # Check if the upload was successful
        # Manually flush to send any remaining documents in the buffer
        batch_client.flush()
        
        print(f"Uploaded {len(documents)} documents in total")
    else:
        print("No documents to upload.")
except HttpResponseError as e:
    print(f"An error occurred: {e}")
    raise  # Re-raise the exception to ensure it errors out
finally:
    # Clean up resources
    batch_client.close()

## 测试搜索
数据上传完成后，我们将在下方本地测试向量相似度搜索和混合搜索，确保其按预期工作。

你可以同时测试纯向量搜索和混合搜索。纯向量搜索会向下面的 `search_text` 传入 `None`，仅基于向量相似度进行搜索。混合搜索则会将查询文本 `query` 传给 `search_text`，把传统的基于关键词的搜索能力与基于向量的相似度搜索结合起来，以提供更相关、更贴合上下文的结果。

In [ ]:
query = "What model should I use to embed?"
# Note: we'll have the GPT choose the category automatically once we put it in ChatGPT
category ="models"

search_client = SearchClient(search_service_endpoint, index_name, AzureKeyCredential(search_service_api_key))
vector_query = VectorizedQuery(vector=generate_embeddings(query, embeddings_model), k_nearest_neighbors=3, fields="content_vector")
  
results = search_client.search(  
    search_text=None, # Pass in None if you want to use pure vector search, and `query` if you want to use hybrid search
    vector_queries= [vector_query], 
    select=["title", "text"],
    filter=f"category eq '{category}'" 
)

for result in results:  
    print(result)


## 创建 Azure 函数

Azure Functions 是在新的 AI 搜索之上构建 API 的一种简便方式。我们的代码（参见本文件夹中的 `function_app.py` 文件，或[此处](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/rag-quickstart/azure/function_app.py)的链接）完成以下工作：

1. 接收以下输入：用户的查询、搜索索引端点、索引名称、k_nearest_neighbors*、要使用的搜索列（content_vector 或 title_vector），以及是否使用混合查询
2. 对用户的查询进行嵌入。
3. 执行向量搜索并检索相关的文本块。
4. 将这些相关文本块作为响应体返回。

*在向量搜索的语境下，k_nearest_neighbors 指定搜索应返回的“最接近”向量（以余弦相似度衡量）的数量。例如，如果 k_nearest_neighbors 设为 3，搜索将返回索引中与查询向量最相似的 3 个向量。

> 请注意，该 Azure Function _没有任何身份验证_。不过，你可以按照[这里](https://learn.microsoft.com/en-us/azure/azure-functions/security-concepts?tabs=v4)的文档为其设置身份验证

### 创建存储账户

我们可以使用下面的代码创建一个新的存储账户，你也可以跳过该代码块，并修改后续步骤以使用现有的存储账户。这最多可能需要 30 秒。

In [ ]:
## Update below with a different name
storage_account_name = "<enter-storage-account-name>"

## Use below SKU or any other SKU as per your requirement
sku = "Standard_LRS"
resource_client = ResourceManagementClient(credential, subscription_id)
storage_client = StorageManagementClient(credential, subscription_id)

# Create resource group if it doesn't exist
rg_result = resource_client.resource_groups.create_or_update(resource_group, {"location": region})

# Create storage account
storage_async_operation = storage_client.storage_accounts.begin_create(
    resource_group,
    storage_account_name,
    {
        "sku": {"name": sku},
        "kind": "StorageV2",
        "location": region,
    },
)
storage_account = storage_async_operation.result()

print(f"Storage account {storage_account.name} created")


### 创建 Function App
当通过 GPT Action 触发时，Python 代码就会在这个 Function App 中执行。要进一步了解 Function App，请参阅[这里](https://learn.microsoft.com/en-us/azure/azure-functions/functions-overview?pivots=programming-language-csharp)的文档。

要部署 Function App，我们需要使用 Azure CLI 和 Azure Functions Core Tools。

> 下面的代码会尝试根据你的平台类型，在虚拟环境中安装并运行这些工具；如果不起作用，请阅读 Azure 文档，了解如何安装 [Azure Function Core Tools](https://learn.microsoft.com/en-us/azure/azure-functions/create-first-function-cli-python?tabs=linux,bash,azure-cli,browser) 和 [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli)。完成安装后，在终端中进入本文件夹，然后运行下面的 `subprocess.run` 命令。

首先，我们要确保环境中具备运行所需 Azure 命令的相关工具。安装可能需要几分钟。

In [ ]:
os_type = platform.system()

if os_type == "Windows":
    # Install Azure Functions Core Tools on Windows
    subprocess.run(["npm", "install", "-g", "azure-functions-core-tools@3", "--unsafe-perm", "true"], check=True)
    # Install Azure CLI on Windows
    subprocess.run(["powershell", "-Command", "Invoke-WebRequest -Uri https://aka.ms/installazurecliwindows -OutFile .\\AzureCLI.msi; Start-Process msiexec.exe -ArgumentList '/I AzureCLI.msi /quiet' -Wait"], check=True)
elif os_type == "Darwin":  # MacOS
    # Install Azure Functions Core Tools on MacOS
    if platform.machine() == 'arm64':
        # For M1 Macs
        subprocess.run(["arch", "-arm64", "brew", "install", "azure-functions-core-tools@3"], check=True)
    else:
        # For Intel Macs
        subprocess.run(["brew", "install", "azure-functions-core-tools@3"], check=True)
    # Install Azure CLI on MacOS
    subprocess.run(["brew", "update"], check=True)
    subprocess.run(["brew", "install", "azure-cli"], check=True)
elif os_type == "Linux":
    # Install Azure Functions Core Tools on Linux
    subprocess.run(["curl", "https://packages.microsoft.com/keys/microsoft.asc", "|", "gpg", "--dearmor", ">", "microsoft.gpg"], check=True, shell=True)
    subprocess.run(["sudo", "mv", "microsoft.gpg", "/etc/apt/trusted.gpg.d/microsoft.gpg"], check=True)
    subprocess.run(["sudo", "sh", "-c", "'echo \"deb [arch=amd64] https://packages.microsoft.com/repos/microsoft-ubuntu-$(lsb_release -cs)-prod $(lsb_release -cs) main\" > /etc/apt/sources.list.d/dotnetdev.list'"], check=True, shell=True)
    subprocess.run(["sudo", "apt-get", "update"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "azure-functions-core-tools-3"], check=True)
    # Install Azure CLI on Linux
    subprocess.run(["curl", "-sL", "https://aka.ms/InstallAzureCLIDeb", "|", "sudo", "bash"], check=True, shell=True)
else:
    # Raise an error if the operating system is not supported
    raise OSError("Unsupported operating system")

# Verify the installation of Azure Functions Core Tools
subprocess.run(["func", "--version"], check=True)
# Verify the installation of Azure CLI
subprocess.run(["az", "--version"], check=True)

subprocess.run([
    "az", "login"
], check=True)

现在，我们需要创建一个 `local.settings.json` 文件，其中包含 Azure 的关键环境变量

In [ ]:
local_settings_content = f"""
{{
  "IsEncrypted": false,
  "Values": {{
    "AzureWebJobsStorage": "UseDevelopmentStorage=true",
    "FUNCTIONS_WORKER_RUNTIME": "python",
    "OPENAI_API_KEY": "{openai_api_key}",
    "EMBEDDINGS_MODEL": "{embeddings_model}",
    "SEARCH_SERVICE_API_KEY": "{search_service_api_key}",
  }}
}}
"""

with open("local.settings.json", "w") as file:
    file.write(local_settings_content)

检查 `local.settings.json` 文件，确保其中的环境变量符合你的预期。

现在，在下方为你的应用命名，然后就可以创建 Function App 并发布函数了。

In [ ]:
# Replace this with your own values. This name will appear in the URL of the API call https://<app_name>.azurewebsites.net
app_name = "<app-name>"

subprocess.run([
    "az", "functionapp", "create",
    "--resource-group", resource_group,
    "--consumption-plan-location", region,
    "--runtime", "python",
    "--name", app_name,
    "--storage-account", storage_account_name,
    "--os-type", "Linux",
], check=True)

创建 Function App 之后，我们需要向其中添加配置变量，以便在函数中使用。具体来说，我们需要 `OPENAI_API_KEY`、`SEARCH_SERVICE_API_KEY` 和 `EMBEDDINGS_MODEL`，因为 `function_app.py` 代码中都会用到它们。

In [ ]:
# Collect the relevant environment variables 
env_vars = {
    "OPENAI_API_KEY": openai_api_key,
    "SEARCH_SERVICE_API_KEY": search_service_api_key,
    "EMBEDDINGS_MODEL": embeddings_model
}

# Create the settings argument for the az functionapp create command
settings_args = []
for key, value in env_vars.items():
    settings_args.append(f"{key}={value}")

subprocess.run([
    "az", "functionapp", "config", "appsettings", "set",
    "--name", app_name,
    "--resource-group", resource_group,
    "--settings", *settings_args
], check=True)

现在我们可以将函数代码 `function_app.py` 发布到 Azure Function 了。部署最多可能需要 10 分钟。完成后，我们就拥有了一个基于 Azure AI Search、通过 Azure Function 提供的 API 端点。

In [ ]:
subprocess.run([
    "func", "azure", "functionapp", "publish", app_name
], check=True)

## 在 ChatGPT 的自定义 GPT 中接入
现在我们已经有了一个查询该向量搜索索引的 Azure Function，接下来把它设置为 GPT Action！

关于 GPT，请参阅[这里](https://openai.com/index/introducing-gpts/)的文档；关于 GPT Actions，请参阅[这里](https://platform.openai.com/docs/actions)。将下面的内容分别用作 GPT 的指令和 GPT Action 的 OpenAPI 规范。


### 创建 OpenAPI 规范
下面是一个 OpenAPI 规范示例。运行下面的代码块后，一份可用的规范会被复制到剪贴板，供你粘贴到 GPT Action 中。

请注意，默认情况下它没有任何身份验证，但你可以参照[这篇 cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function#part-2-set-up-auth) 中 Authentication 部分的模式，或查阅[这里](https://learn.microsoft.com/en-us/azure/app-service/overview-authentication-authorization)的文档，为 Azure Functions 设置 OAuth。

In [ ]:

spec = f"""
openapi: 3.1.0
info:
  title: Vector Similarity Search API
  description: API for performing vector similarity search.
  version: 1.0.0
servers:
  - url: https://{app_name}.azurewebsites.net/api
    description: Main (production) server
paths:
  /vector_similarity_search:
    post:
      operationId: vectorSimilaritySearch
      summary: Perform a vector similarity search.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                search_service_endpoint:
                  type: string
                  description: The endpoint of the search service.
                index_name:
                  type: string
                  description: The name of the search index.
                query:
                  type: string
                  description: The search query.
                k_nearest_neighbors:
                  type: integer
                  description: The number of nearest neighbors to return.
                search_column:
                  type: string
                  description: The name of the search column.
                use_hybrid_query:
                  type: boolean
                  description: Whether to use a hybrid query.
                category:
                  type: string
                  description: category to filter.
              required:
                - search_service_endpoint
                - index_name
                - query
                - k_nearest_neighbors
                - search_column
                - use_hybrid_query
      responses:
        '200':
          description: A successful response with the search results.
          content:
            application/json:
              schema:
                type: object
                properties:
                  results:
                    type: array
                    items:
                      type: object
                      properties:
                        id:
                          type: string
                          description: The identifier of the result item.
                        score:
                          type: number
                          description: The similarity score of the result item.
                        content:
                          type: object
                          description: The content of the result item.
        '400':
          description: Bad request due to missing or invalid parameters.
        '500':
          description: Internal server error.
"""
pyperclip.copy(spec)
print("OpenAPI spec copied to clipboard")
print(spec)

### 创建 GPT 指令

你可以根据需要随意修改指令。关于提示词工程的一些技巧，请查看我们[这里](https://platform.openai.com/docs/guides/prompt-engineering)的文档。

In [ ]:
instructions = f'''
You are an OAI docs assistant. You have an action in your knowledge base where you can make a POST request to search for information. The POST request should always include: {{
    "search_service_endpoint": "{search_service_endpoint}",
    "index_name": {index_name},
    "query": "<user_query>",
    "k_nearest_neighbors": 1,
    "search_column": "content_vector",
    "use_hybrid_query": true,
    "category": "<category>"
}}. Only the query and category change based on the user's request. Your goal is to assist users by performing searches using this POST request and providing them with relevant information based on the query.

You must only include knowledge you get from your action in your response.
The category must be from the following list: {categories}, which you should determine based on the user's query. If you cannot determine, then do not include the category in the POST request.
'''
pyperclip.copy(instructions)
print("GPT Instructions copied to clipboard")
print(instructions)

现在我们拥有了一个可以查询向量数据库的 GPT！

# 回顾
通过以下步骤，我们已经成功地将 Azure AI Search 与 ChatGPT 中的 GPT Actions 集成：
1. 使用 OpenAI 的嵌入对文档进行嵌入，同时使用 gpt-4o 添加一些额外的元数据。
2. 将这些数据上传到 Azure AI Search。
3. 使用 Azure Functions 创建了一个用于查询的端点。
4. 将其整合到自定义 GPT 中。

现在，我们的 GPT 可以检索信息来帮助回答用户的问题，使回答更加准确，并且更贴合我们的数据。下面是 GPT 的实际运行效果：

# ![azure-rag-quickstart-gpt.png](../../../../images/azure-rag-quickstart-gpt.png)